# Colab install

In [ ]:
!pip install "stable-baselines3[extra]"
!pip install tetris-gymnasium
!pip install gymnasium[other]

In [ ]:
file_path = r'S:\Universidad\Juegos_IA\.venv\Lib\site-packages\tetris_gymnasium\wrappers\observation.py'

with open(file_path, 'r') as f:
    content = f.read()
modified_content = content.replace("high=len(env.unwrapped.tetrominoes)", "high=255")
modified_content = modified_content.replace("self.render_scaling_factor", "20")

with open(file_path, 'w') as f:
    f.write(modified_content)

with open(file_path, 'r') as f:
    print(f.read())


In [ ]:
file_path = r'S:\Universidad\Juegos_IA\.venv\Lib\site-packages\tetris_gymnasium\wrappers\grouped.py'

with open(file_path, 'r') as f:
    content = f.read()
modified_content = content.replace("high=env.unwrapped.height * env.unwrapped.width", "high=255")
#modified_content = modified_content.replace("self.render_scaling_factor", "20")
#modified_content = modified_content.replace("dtype=np.float32", "dtype=np.uint8")

with open(file_path, 'w') as f:
    f.write(modified_content)

with open(file_path, 'r') as f:
    print(f.read())


# Config

In [ ]:
RENDER_ENV = False
LOAD_MODEL = False
new_size = (84,84) #(84,84) (96,136)
batch_size = 32
num_episodes = 9360 #56160 12 horas #4680 para 1 hora pc Seba
max_episode_steps = 500
fase1=1.5e6
fase2=2.5e6
fase3=6e6
total_steps = fase1+fase2+fase3
num_stacked_frames = 4
intervals = 4
Model = "DQN" # DQN o PPO
version = 10

#Hiperparametros Compartidos
LR = 2e-5 #2e-5
GAMMA = 0.999 #0.999
BATCH_SIZE=256 #256

#DQN Hiperparametros
EXPLORATION = 0.3
BUFFER_SIZE = 20000

#PPO Hiperparametros
N_STEPS = 4096
ENT_COEF = 0.01 
CLIP_RANGE = 0.2 #0.1


# imports

In [ ]:
import importlib
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, FrameStackObservation, GrayscaleObservation
from stable_baselines3 import DQN, PPO
import os
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_util import make_vec_env

# Funciones

In [ ]:
def get_last_modified_file(directory_path):
    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist.")
        return None
    files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
    if not files:
        return None
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

target_directory = f"../Models_Saves/{Model}"  
model_load_path = get_last_modified_file(target_directory)

if model_load_path:
    print(f"The last modified file is: {model_load_path}")
else:
    print("No files found in the directory or directory does not exist.")

In [ ]:
try:
    os.mkdir("../Models_Saves")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO/checkpoint")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN/checkpoint")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Logs")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Logs/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Logs/DQN")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
def calc_max_height(mat):
  mat1 = np.rot90(np.rot90(mat))
  for i in range(len(mat1)):
    count = 0
    for col in mat1[i]:
      if col > 0:
        count+=1
    if count == 0:
      return i-1
  return 19

def calc_holes(mat,height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  holes = 0
  for row in mat1:
    for i in range(height+1):
      if row[i] == 0:
        holes += 1
  return holes

def calc_adj_col(mat, height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  prev_height = -1
  dif_count = 0
  for row in mat1:
    act_height = 0
    for i in range(height+1):
      if row[i] != 0:
        act_height = i+1
    if prev_height >= 0:
      dif_count+=(abs(prev_height-act_height))
    prev_height = act_height
  return dif_count


In [ ]:

class CustomRewardWrapper(gym.Wrapper):
    #def __init__(self, env, holes_penalty = 0.005, height_penalty = 0.05, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.03):
    #Prometedor pero algo falta def __init__(self, env, holes_penalty = 0.000002, height_penalty = 0.00002, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.00001):
    def __init__(self, env, increase_height_penalty = 1.0, increase_holes_penalty = 1.0, dif_heigh_penalty = 0.2, line1=0.5, line2=3, line3=6, line4=15, gameover=50, trunc=10): #, holes_penalty = 0.000002, height_penalty = 0.00002
        super(CustomRewardWrapper, self).__init__(env)
        #self.holes_penalty = holes_penalty
        #self.height_penalty = height_penalty
        self.increase_height_penalty = increase_height_penalty
        self.increase_holes_penalty = increase_holes_penalty
        self.dif_heigh_penalty = dif_heigh_penalty
        self.line1 = line1
        self.line2 = line2
        self.line3 = line3
        self.line4 = line4
        self.gameover = gameover
        self.trunc = trunc

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables)
        self.previous_holes = calc_holes(game_variables, self.previous_max_height)
        self.previous_bumps = calc_adj_col(game_variables, self.previous_max_height)

        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward

        match int(reward):
            case 1:
                custom_reward+=self.line1
            case 2:
                custom_reward+=self.line2
            case 3:
                custom_reward+=self.line3
            case 4:
                custom_reward+=self.line4
        
        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            # Calcula la altura maxima actual
            current_max_height = calc_max_height(game_variables)
            # Calcula ls espacios vacios entre la base y la altura
            current_holes = calc_holes(game_variables, current_max_height)
            #Calcula la diferencia de altura entre las columnas adyacentes y las suma
            current_height_dif = calc_adj_col(game_variables, current_max_height)
            # #Penalizacion constante por altura
            # custom_reward -= current_max_height*self.height_penalty
            # #Penalizacion constante por agujeros
            # custom_reward -= current_holes*self.holes_penalty
            # #Penalizacion constante por diferencia de altura entre columnas
            # custom_reward -= current_height_dif*self.dif_heigh_penalty
            # Penalizacion y recompensa por aumentar altura o disminuir altura respectivamente
            bumps_change = current_height_dif-self.previous_bumps
            if bumps_change > 0:
                custom_reward -= bumps_change*self.dif_heigh_penalty
            height_change=current_max_height-self.previous_max_height
            if height_change > 0:
                custom_reward -= height_change*self.increase_height_penalty
            #Penalizacion y recompensa por aumentar agujeros o disminuir agujeros respectivamente
            holes_change = current_holes-self.previous_holes
            if holes_change > 0:
                custom_reward -= holes_change*self.increase_holes_penalty
            elif holes_change < 0:
                custom_reward += abs(holes_change)*self.increase_holes_penalty
            #Actualiza valores previos para el siguiente paso
            self.previous_holes = current_holes
            self.previous_max_height = current_max_height
            self.previous_bumps = current_height_dif
        if terminated:
            custom_reward -= self.gameover
        elif truncated:
            custom_reward += self.trunc
        
        return obs, custom_reward, terminated, truncated, info

# Make single ENV

In [ ]:
def make_env(*, game, obs_type, render_mode, record = True, increase_height_penalty = 1.0, increase_holes_penalty = 1.0, dif_heigh_penalty = 0.2, line1=0.5, line2=3, line3=6, line4=15, gameover=50, trunc=10, fase, **kwargs):
    env = gym.make(game,render_mode=render_mode, **kwargs)
    env = TimeLimit(env, max_episode_steps=max_episode_steps)
    if obs_type == "RGB":
        env = RgbObservation(env)
        env = ResizeObservation(env, new_size)
        env = GrayscaleObservation(env)
        env = FrameStackObservation(env, stack_size=num_stacked_frames)
    elif obs_type == "FV":
        env = FeatureVectorObservation(env)
        env = FrameStackObservation(env, stack_size=num_stacked_frames)
    if record:
      env = Monitor(env, f'../Logs/{Model}/{Model}_V{version}_S{total_steps}_Fase{fase}.csv')
    env = CustomRewardWrapper(env, increase_height_penalty = increase_height_penalty, increase_holes_penalty = increase_holes_penalty, dif_heigh_penalty = dif_heigh_penalty, line1=line1, line2=line2, line3=line3, line4=line4, gameover=gameover, trunc=trunc)
    checkpoint_callback = CheckpointCallback(
        save_freq=100000,
        save_path=f'../Models_Saves/{Model}/checkpoint',
        name_prefix=f'{Model}_tetris_step'
    )
    env.reset(seed=42)
    return env

# Training RGB

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="RGB", increase_height_penalty = 1.5, increase_holes_penalty = 1.5, dif_heigh_penalty = 0.05, line1=0.5, line2=0.5, line3=0.5, line4=0.5, gameover=50, trunc=10)
if Model == "DQN":
    model = DQN("CnnPolicy", env, buffer_size=BUFFER_SIZE, verbose=1, exploration_fraction=EXPLORATION, exploration_final_eps = 0.05, learning_rate=LR, batch_size=BATCH_SIZE, gamma=GAMMA) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
else:
    model = PPO("CnnPolicy", env, verbose=1, n_steps=N_STEPS, clip_range=CLIP_RANGE, ent_coef=ENT_COEF, gamma=GAMMA, batch_size=BATCH_SIZE, learning_rate=LR) # para Mlp usar FeatureVectorObservation para Cnn

model.learn(total_timesteps=fase1, log_interval=4)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="RGB", increase_height_penalty = 1.0, increase_holes_penalty = 2.0, dif_heigh_penalty = 0.05, line1=0.5, line2=0.5, line3=0.5, line4=0.5, gameover=50, trunc=10)
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 
    model.exploration_rate = 0.3
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 

model.learn(total_timesteps=fase2, log_interval=4,reset_num_timesteps=False)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="RGB", increase_height_penalty = 1.0, increase_holes_penalty = 1.0, dif_heigh_penalty = 0.2, line1=0.5, line2=3, line3=6, line4=15, gameover=50, trunc=10)
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 
    model.exploration_rate = 0.3
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 

model.learn(total_timesteps=fase3, log_interval=4, reset_num_timesteps=False)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

# Recording

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", record = False, obs_type="FV", fase = 4) #Cambiar obs_type entre FV RGB dependiendo del modelo entrenado
try:
  env = RecordVideo(
    env,
    video_folder=f'../Video_Tetris_IA/{Model}',   
    name_prefix=f'{Model}_eval-V{version}-Trained_steps_{total_steps}',         
    episode_trigger=lambda x: True    
  )
except Exception as e:
  print(f'error implementando grabacion: {e}')
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True)
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True)

In [ ]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state)#, deterministic=True
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

# Feature Vector Training

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="FV", increase_height_penalty = 1.5, increase_holes_penalty = 1.5, dif_heigh_penalty = 0.05, line1=0.5, line2=0.5, line3=0.5, line4=0.5, gameover=50, trunc=10, fase=1)
if Model == "DQN":
    model = DQN("MlpPolicy", env, buffer_size=BUFFER_SIZE, verbose=1, exploration_fraction=EXPLORATION, exploration_final_eps = 0.05, learning_rate=LR, batch_size=BATCH_SIZE, gamma=GAMMA) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
else:
    model = PPO("MlpPolicy", env, verbose=1, n_steps=N_STEPS, clip_range=CLIP_RANGE, ent_coef=ENT_COEF, gamma=GAMMA, batch_size=BATCH_SIZE, learning_rate=LR) # para Mlp usar FeatureVectorObservation para Cnn

model.learn(total_timesteps=fase1, log_interval=4)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="FV", increase_height_penalty = 1.0, increase_holes_penalty = 2.0, dif_heigh_penalty = 0.05, line1=0.5, line2=0.5, line3=0.5, line4=0.5, gameover=50, trunc=10, fase=2)
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 
    model.exploration_rate = 0.3
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 

model.learn(total_timesteps=fase2, log_interval=4,reset_num_timesteps=False)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="FV", increase_height_penalty = 1.0, increase_holes_penalty = 1.0, dif_heigh_penalty = 0.2, line1=0.5, line2=3, line3=6, line4=15, gameover=50, trunc=10, fase=3)
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 
    model.exploration_rate = 0.3
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}", env=env, print_system_info=True) 

model.learn(total_timesteps=fase3, log_interval=4, reset_num_timesteps=False)
model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{total_steps}")
env.close()

# Graficar

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
df1 = pd.read_csv(f'../Logs/../Logs/PPO/PPO_V{version}_S{total_steps}_Fase1.csv.monitor.csv', header=1)
df2 = pd.read_csv(f'../Logs/../Logs/PPO/PPO_V{version}_S{total_steps}_Fase2.csv.monitor.csv', header=1)
df3 = pd.read_csv(f'../Logs/../Logs/PPO/PPO_V{version}_S{total_steps}_Fase3.csv.monitor.csv', header=1)
df = pd.concat([df1,df2,df3], ignore_index=True)
plt.figure(figsize=(10, 5))
plt.plot(df['l'], label='Episode Length', alpha=0.5)

window_size = 50  
df['smooth_reward'] = df['l'].rolling(window=window_size).mean()
plt.plot(df['smooth_reward'], label=f'Smooth Length', linestyle='--')

plt.xlabel('Episode')
plt.ylabel('Length')
plt.title('Length Graph')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
df1 = pd.read_csv(f'../Logs/../Logs/DQN/DQN_V{version}_S{total_steps}_Fase1.csv.monitor.csv', header=1)
df2 = pd.read_csv(f'../Logs/../Logs/DQN/DQN_V{version}_S{total_steps}_Fase2.csv.monitor.csv', header=1)
df3 = pd.read_csv(f'../Logs/../Logs/DQN/DQN_V{version}_S{total_steps}_Fase3.csv.monitor.csv', header=1)
df = pd.concat([df1,df2,df3], ignore_index=True)
plt.figure(figsize=(10, 5))
plt.plot(df['l'], label='Episode Length', alpha=0.5)

window_size = 50
df['smooth_reward'] = df['l'].rolling(window=window_size).mean()
plt.plot(df['smooth_reward'], label=f'Smooth Length', linestyle='--')

plt.xlabel('Episode')
plt.ylabel('Length')
plt.title('Length Graph')
plt.grid(True)
plt.legend()
plt.show()